In [1]:
import pandas as pd

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"


Download our data

In [3]:
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [4]:
row = df.iloc[0]
row


PULocationID                             43
DOLocationID                            186
trip_distance                          1.68
total_amount                          22.15
tpep_pickup_datetime    2025-11-01 00:13:25
Name: 0, dtype: object

Define a dataclass for our message. This gives us a clear schema for each taxi trip

In [7]:
from dataclasses import dataclass

In [23]:
@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int


In [24]:
ride = Ride(
    PULocationID=1,
    DOLocationID=10,
    trip_distance=10,
    total_amount=20,
    tpep_pickup_datetime=1761956005000
)

This converts our data into a string in json format

In [25]:
import json

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

Next connect to kafka

The `bootstrap_servers` is where the broker accepts
connections - `localhost:9092` because we're running this from our laptop
(outside Docker). 

In a system like Redpanda or Kafka, `the broker` is the server that acts as the middleman between the people sending data and the people receiving it

In [21]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer 
)

In [27]:
import dataclasses

#save ride class as dictionary
ride_dict = dataclasses.asdict(ride)
#convert dict to json 
json.dumps(ride_dict).encode('utf-8')


b'{"PULocationID": 1, "DOLocationID": 10, "trip_distance": 10, "total_amount": 20, "tpep_pickup_datetime": 1761956005000}'

In [29]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=ride_dict)
producer.flush()